## Import

In [1]:
import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, log_loss, average_precision_score)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
N_TRIALS = 30   # same Optuna budget for every model
INNER_K = 3     # inner folds used for tuning
os.makedirs("weights_V6_0", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
print("cleaned:", df.shape, "| Response rate:", round(df["Response"].mean(), 4))

CLF_NUMERIC = ["AcceptedCmp5","AcceptedCmp3","MntWines","MntMeatProducts","NumCatalogPurchases",
    "Recency","Customer_Tenure_Days","MntGoldProds","Income","MntFruits","AcceptedCmp1",
    "AcceptedCmp2","NumWebPurchases","MntSweetProducts","MntFishProducts","NumStorePurchases","Age"]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {len(FEATURE_COLS)} dims")

cleaned: (2236, 28) | Response rate: 0.1494
features: 17 numeric + 5 dummies = 22 dims


## Dataloader

In [4]:
X_all = X_full.values
y_all = y_full.astype(int)
inc_idx = FEATURE_COLS.index("Income")

def prep_split(tr_idx, te_idx):
    """Median-fill Income and standardise using the training rows only."""
    Xtr, Xte = X_all[tr_idx].copy(), X_all[te_idx].copy()
    med = np.nanmedian(Xtr[:, inc_idx])
    for a in (Xtr, Xte):
        a[:, inc_idx] = np.where(np.isnan(a[:, inc_idx]), med, a[:, inc_idx])
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xte), y_all[tr_idx], y_all[te_idx]

# same 80/20 stratified split as every earlier version (seed 42)
idx_all = np.arange(len(y_all))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED, stratify=y_all)
Xtr, Xte, ytr, yte = prep_split(tr_idx, te_idx)
print(f"train {Xtr.shape} test {Xte.shape} | test pos rate {yte.mean():.3f} | test positives {int(yte.sum())}")

train (1788, 22) test (448, 22) | test pos rate 0.150 | test positives 67


## Model

In [5]:
from catboost import CatBoostClassifier
SPECS = {
 "CatBoost": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(n=t.suggest_int("n", 100, 500, step=100), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         d=t.suggest_int("d", 3, 8), l2=t.suggest_float("l2", 1.0, 20.0, log=True),
                         acw=t.suggest_categorical("acw", [None, "Balanced"])),
    make=lambda p, s: CatBoostClassifier(iterations=p["n"], learning_rate=p["lr"], depth=p["d"], l2_leaf_reg=p["l2"],
                         auto_class_weights=p["acw"], thread_count=1, random_seed=s, verbose=0, allow_writing_files=False)),
}

## Tuning

In [6]:
def score_of(est, X, kind):
    return est.predict_proba(X)[:, 1] if kind == "proba" else est.decision_function(X)

def inner_oof(spec, params, X, y, seed):
    """Out-of-fold scores on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in StratifiedKFold(INNER_K, shuffle=True, random_state=seed).split(X, y):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = score_of(est, X[b], spec["kind"])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) maximising average precision of the inner out-of-fold scores."""
    def objective(trial):
        return average_precision_score(y, inner_oof(spec, spec["space"](trial), X, y, seed))
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=N_TRIALS)
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def best_threshold(y, s):
    """F1-maximising threshold, searched on training-side out-of-fold scores only."""
    cand = np.unique(np.quantile(s, np.linspace(0.40, 0.99, 120)))
    f = [f1_score(y, (s >= t).astype(int), zero_division=0) for t in cand]
    return float(cand[int(np.argmax(f))])

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    """Threshold from inner OOF on (Xa, ya); refit on all of (Xa, ya); score (Xb, yb)."""
    thr = best_threshold(ya, inner_oof(spec, params, Xa, ya, seed))
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    s = score_of(est, Xb, spec["kind"])
    pred = (s >= thr).astype(int)
    row = {"f1": f1_score(yb, pred, zero_division=0), "accuracy": accuracy_score(yb, pred),
           "precision": precision_score(yb, pred, zero_division=0),
           "recall": recall_score(yb, pred, zero_division=0),
           "roc_auc": roc_auc_score(yb, s), "threshold": thr}
    if spec["kind"] == "proba":
        row["bce"] = log_loss(yb, np.clip(s, 1e-12, 1 - 1e-12))
    return row, s

In [7]:
best_params = {}
for name, spec in SPECS.items():
    t0 = time.time()
    best_params[name] = tune(spec, Xtr, ytr, SEED)
    print(f"{name:32s} tuned in {time.time() - t0:6.1f}s | {best_params[name]}")
json.dump(best_params, open("weights_V6_0/classification_catboost_best_params.json", "w"), indent=2, default=str)

CatBoost                         tuned in   33.9s | {'n': 500, 'lr': 0.013920868322942554, 'd': 5, 'l2': 6.099535729196778, 'acw': 'Balanced'}


## Training

In [8]:
results, scores = [], {}
for name, spec in SPECS.items():
    row, s = fit_eval(spec, best_params[name], Xtr, ytr, Xte, yte, SEED)
    scores[name] = s
    results.append(dict(model=name, tier=spec["tier"], selection="inner validation",
                        **{k: round(float(v), 4) for k, v in row.items()}, params=best_params[name]))
    print(f"{name:32s} F1 = {row['f1']:.4f}")

CatBoost                         F1 = 0.5731


## Evaluation

In [9]:
table = pd.DataFrame(results).sort_values("f1", ascending=False)
print(table.drop(columns=["params", "selection"]).to_string(index=False))
json.dump({"n_test": int(len(yte)), "seed": SEED, "n_trials": N_TRIALS, "inner_k": INNER_K, "models": results},
          open("weights_V6_0/classification_catboost_test_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V6_0/classification_catboost_test_scores.npz", y_true=yte, **{k: v for k, v in scores.items()})

   model       tier     f1  accuracy  precision  recall  roc_auc  threshold    bce
CatBoost curriculum 0.5731    0.8371     0.4712  0.7313     0.88     0.5071 0.3733
